In [1]:
import os, json
import numpy as np
import pandas as pd

CANDIDATE_PATHS = [
    r"C:\Users\DELL\Desktop\IBM\PROJECT TRUST SHIELD\PaySim Data.csv",
    r"C:\Users\DELL\Desktop\IBM\PAYSIM DATASET.csv",
]
dataset_path = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)
if dataset_path is None:
    raise FileNotFoundError("Dataset not found. Edit CANDIDATE_PATHS above.")
print("Using file:", dataset_path)

# nameOrig, nameDest and isFlaggedFraud are deliberately NOT loaded:
#  - IDs are not usable features here
#  - isFlaggedFraud is the legacy rule's output (using it would be cheating)
cols = ['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
        'oldbalanceDest', 'newbalanceDest', 'isFraud']
dtypes = {'step': np.int32, 'type': 'category', 'amount': np.float32,
          'oldbalanceOrg': np.float32, 'newbalanceOrig': np.float32,
          'oldbalanceDest': np.float32, 'newbalanceDest': np.float32,
          'isFraud': np.int8}

# Whole file (no nrows): the first rows only cover the earliest steps
df = pd.read_csv(dataset_path, usecols=cols, dtype=dtypes)
print("All rows:", df.shape[0], "| step range:", df['step'].min(), "to", df['step'].max())

df = df[df['type'].isin(['CASH_OUT', 'TRANSFER'])].copy()
df['type'] = df['type'].astype(str)
print("CASH_OUT + TRANSFER rows:", df.shape[0])
print("Fraud rows:", int(df['isFraud'].sum()), "| fraud rate:", round(df['isFraud'].mean() * 100, 4), "%")

Using file: C:\Users\DELL\Desktop\IBM\PROJECT TRUST SHIELD\PaySim Data.csv
All rows: 6362620 | step range: 1 to 743
CASH_OUT + TRANSFER rows: 2770409
Fraud rows: 8213 | fraud rate: 0.2965 %


In [2]:
# Only fields that exist BEFORE a transaction is approved
df['is_transfer'] = (df['type'] == 'TRANSFER').astype(np.int8)
df['amount_to_old_balance'] = (df['amount'] / (df['oldbalanceOrg'] + 1.0)).astype(np.float32)
df['amount_exceeds_balance'] = (df['amount'] > df['oldbalanceOrg']).astype(np.int8)

# Post-transaction fields: kept ONLY to measure how much they leak (Cell 4)
df['reconciliation_anomaly'] = (df['amount'] - (df['newbalanceDest'] - df['oldbalanceDest'])).abs().astype(np.float32)

HONEST_FEATURES = ['is_transfer', 'amount', 'oldbalanceOrg', 'oldbalanceDest',
                   'amount_to_old_balance', 'amount_exceeds_balance']
LEAKY_EXTRA = ['newbalanceOrig', 'newbalanceDest', 'reconciliation_anomaly']

# Split by time (step): early = train, middle = validation, latest = test
s_max = int(df['step'].max())
cut_train = int(s_max * 0.65)
cut_val = int(s_max * 0.80)
train = df[df['step'] <= cut_train]
val = df[(df['step'] > cut_train) & (df['step'] <= cut_val)]
test = df[df['step'] > cut_val]

for name, part in [('train', train), ('validation', val), ('test', test)]:
    print(f"{name:10s} steps {int(part['step'].min())}-{int(part['step'].max())} | rows {len(part):>9,} | "
          f"fraud {int(part['isFraud'].sum()):>5,} | rate {part['isFraud'].mean() * 100:.4f}%")
    assert part['isFraud'].sum() > 0, f"No fraud rows in {name}: move the cut-offs"

train      steps 1-482 | rows 2,636,285 | fraud 5,379 | rate 0.2040%
validation steps 483-594 | rows    82,838 | fraud 1,180 | rate 1.4245%
test       steps 595-743 | rows    51,286 | fraud 1,654 | rate 3.2251%



LightGBM Model Initialization & Training

This applies the optimized class balancing calculation to protect regular users and drives the training loop down to less than 20 seconds.

In [3]:
import warnings
warnings.filterwarnings("ignore", message=".*eval_set.*")
import lightgbm as lgb
from sklearn.metrics import roc_curve

def fit_model(features):
    pos = int(train['isFraud'].sum())
    neg = len(train) - pos
    m = lgb.LGBMClassifier(
        objective='binary', metric='average_precision',
        n_estimators=600, learning_rate=0.05, num_leaves=31, max_depth=6,
        scale_pos_weight=(neg / pos) * 0.1,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbose=-1)
    m.fit(train[features], train['isFraud'],
          eval_set=[(val[features], val['isFraud'])],
          callbacks=[lgb.early_stopping(50, first_metric_only=True, verbose=False)])
    return m

def pick_threshold(y, p, target_fpr):
    fpr, tpr, thr = roc_curve(y, p)
    ok = fpr <= target_fpr          # fpr[0] is 0, so this is never empty
    return float(thr[ok][-1])       # lowest cut-off that still meets the target

def metrics(y, pred):
    y = np.asarray(y); pred = np.asarray(pred).astype(bool)
    tp = int((pred & (y == 1)).sum()); fp = int((pred & (y == 0)).sum())
    fn = int((~pred & (y == 1)).sum()); tn = int((~pred & (y == 0)).sum())
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn, 'recall': round(recall, 4),
            'precision': round(precision, 4), 'FPR': round(fpr, 5), 'F1': round(f1, 4)}

print("[DS1] Training LightGBM on TRAIN only (early stopping on VALIDATION)...")
model = fit_model(HONEST_FEATURES)
print("Boosting rounds used:", model.best_iteration_)

p_val = model.predict_proba(val[HONEST_FEATURES])[:, 1]
y_val = val['isFraud'].values

print("\nTRADE-OFF TABLE (VALIDATION data, used to choose the threshold):")
rows = []
for t in [0.001, 0.002, 0.005, 0.01]:
    thr = pick_threshold(y_val, p_val, t)
    r = metrics(y_val, p_val >= thr)
    r.update({'target_FPR': t, 'threshold': round(thr, 5),
              'holds_per_1000': round(1000 * (r['TP'] + r['FP']) / len(y_val), 2)})
    rows.append(r)
print(pd.DataFrame(rows)[['target_FPR', 'threshold', 'recall', 'precision', 'FPR', 'F1', 'holds_per_1000']].to_string(index=False))

TARGET_FPR = 0.005   # YOUR decision: change it after reading the table above
threshold = pick_threshold(y_val, p_val, TARGET_FPR)
print("\nChosen target FPR:", TARGET_FPR, "-> threshold:", round(threshold, 5))

# TEST set: used once, at the chosen threshold
p_test = model.predict_proba(test[HONEST_FEATURES])[:, 1]
y_test = test['isFraud'].values
final = metrics(y_test, p_test >= threshold)
print("\nFINAL TEST RESULTS (honest features, time-based split):")
for k, v in final.items():
    print(f"  {k}: {v}")
print("  test rows:", len(y_test), "| fraud cases:", int(y_test.sum()))

[DS1] Training LightGBM on TRAIN only (early stopping on VALIDATION)...
Boosting rounds used: 1

TRADE-OFF TABLE (VALIDATION data, used to choose the threshold):
 target_FPR  threshold  recall  precision     FPR     F1  holds_per_1000
      0.001    1.00000  0.8992     0.9725 0.00037 0.9344           13.17
      0.002    1.00000  0.8992     0.9725 0.00037 0.9344           13.17
      0.005    0.00446  1.0000     0.8121 0.00334 0.8963           17.54
      0.010    0.00232  1.0000     0.6782 0.00686 0.8082           21.00

Chosen target FPR: 0.005 -> threshold: 0.00446

FINAL TEST RESULTS (honest features, time-based split):
  TP: 1654
  FP: 215
  FN: 0
  TN: 49417
  recall: 1.0
  precision: 0.885
  FPR: 0.00433
  F1: 0.939
  test rows: 51286 | fraud cases: 1654



Unvarnished Performance Matrix & Export

This prints out your True Positive, False Positive, and Precision-Recall metrics, then serializes the model file for the SWE handoff loop.

In [4]:
import subprocess, datetime

print("CHECK 1: the 0.5 default threshold, for contrast only (TEST data)")
print(" ", metrics(y_test, p_test >= 0.5))

print("\nCHECK 2: how much do post-transaction fields inflate the scores? (same split, same method)")
leaky_features = HONEST_FEATURES + LEAKY_EXTRA
leaky_model = fit_model(leaky_features)
pv_l = leaky_model.predict_proba(val[leaky_features])[:, 1]
thr_l = pick_threshold(y_val, pv_l, TARGET_FPR)
pt_l = leaky_model.predict_proba(test[leaky_features])[:, 1]
comparison = pd.DataFrame([
    {'features': 'honest (pre-approval only)', **metrics(y_test, p_test >= threshold)},
    {'features': 'leaky (adds post-transaction fields)', **metrics(y_test, pt_l >= thr_l)},
])
print(comparison[['features', 'recall', 'precision', 'FPR', 'F1', 'TP', 'FP', 'FN']].to_string(index=False))

print("\nCHECK 3: feature importance (honest model, split gain)")
imp = pd.Series(model.booster_.feature_importance(importance_type='gain'), index=HONEST_FEATURES)
print((imp / imp.sum()).round(4).sort_values(ascending=False).to_string())

ART = 'artifacts'
os.makedirs(ART, exist_ok=True)
model.booster_.save_model(f'{ART}/model.txt')          # native LightGBM file

meta = {'features': HONEST_FEATURES, 'threshold': threshold, 'target_fpr': TARGET_FPR,
        'split_steps': {'train_max': cut_train, 'val_max': cut_val, 'test_max': s_max},
        'best_iteration': int(model.best_iteration_),
        'final_test_metrics': final}
with open(f'{ART}/model_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

out = test[['step', 'type', 'amount']].copy()
out['y_true'] = y_test
out['score'] = p_test
out['pred_hold'] = (p_test >= threshold).astype(int)
out.to_csv(f'{ART}/test_predictions.csv', index=False)
comparison.to_csv(f'{ART}/honest_vs_leaky.csv', index=False)

example = test[HONEST_FEATURES].iloc[0].to_dict()
booster = lgb.Booster(model_file=f'{ART}/model.txt')
example_score = float(booster.predict(pd.DataFrame([example]))[0])
with open(f'{ART}/example_row.json', 'w') as f:
    json.dump({'input': {k: float(v) for k, v in example.items()},
               'expected_score': example_score}, f, indent=2)

same = np.allclose(booster.predict(test[HONEST_FEATURES].head(1000)), p_test[:1000])
print("\nSaved model reloads and matches the in-memory model:", same)
print("Files in", ART, ":", os.listdir(ART))

CHECK 1: the 0.5 default threshold, for contrast only (TEST data)
  {'TP': 1654, 'FP': 158, 'FN': 0, 'TN': 49474, 'recall': 1.0, 'precision': 0.9128, 'FPR': 0.00318, 'F1': 0.9544}

CHECK 2: how much do post-transaction fields inflate the scores? (same split, same method)
                            features  recall  precision     FPR     F1   TP  FP  FN
          honest (pre-approval only)  1.0000     0.8850 0.00433 0.9390 1654 215   0
leaky (adds post-transaction fields)  0.9994     0.9245 0.00272 0.9605 1653 135   1

CHECK 3: feature importance (honest model, split gain)
amount_to_old_balance     0.5036
amount_exceeds_balance    0.4620
oldbalanceOrg             0.0248
oldbalanceDest            0.0071
is_transfer               0.0027
amount                    0.0000

Saved model reloads and matches the in-memory model: True
Files in artifacts : ['example_row.json', 'honest_vs_leaky.csv', 'model.txt', 'model_meta.json', 'README_model_section.md', 'rule_vs_model_tiers.csv', 'team_messag

In [5]:
try:
    commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'],
                                     stderr=subprocess.DEVNULL).decode().strip()
except Exception:
    commit = 'not in a git repo yet: commit, then rerun this cell'
today = datetime.date.today().isoformat()

fraud_by_type = df.groupby('type')['isFraud'].agg(['sum', 'count']).rename(
    columns={'sum': 'fraud', 'count': 'rows'})
type_lines = "\n".join(f"  {t}: {int(r.fraud):,} fraud of {int(r.rows):,} rows"
                       for t, r in fraud_by_type.iterrows())

def part_line(name, part):
    return (f"  {name}: steps {int(part['step'].min())}-{int(part['step'].max())}, "
            f"{len(part):,} rows, {int(part['isFraud'].sum()):,} fraud")

readme = f"""## Model and evaluation (DS1)

**Data:** PaySim synthetic mobile money dataset (Kaggle). License: see the dataset's Kaggle page [LINK AND LICENSE TO BE ADDED BY TEAM].
**Rows used:** CASH_OUT and TRANSFER only: {len(df):,} rows, {int(df['isFraud'].sum()):,} fraud ({df['isFraud'].mean()*100:.4f}%).

**Split (by time step, not random):**
{part_line('train', train)}
{part_line('validation', val)}
{part_line('test', test)}

**Model:** LightGBM, {int(model.best_iteration_)} boosting rounds (early stopping on validation).
**Features (all available before approval):** {', '.join(HONEST_FEATURES)}.
**Excluded:** newbalanceOrig, newbalanceDest and ledger discrepancy, because they exist only after the transaction.
**Threshold:** {threshold:.5f}, chosen on validation to meet a target false-positive rate of {TARGET_FPR*100:.2f}%.

**Test results (scored once):**

| TP | FP | FN | TN | Recall | Precision | FPR | F1 |
|----|----|----|----|--------|-----------|-----|----|
| {final['TP']} | {final['FP']} | {final['FN']} | {final['TN']} | {final['recall']} | {final['precision']} | {final['FPR']} | {final['F1']} |

**Limitations:** PaySim is synthetic and its fraud patterns are simple, so these figures do not represent real-world performance. The test set has {int(y_test.sum())} fraud cases, so rates carry sampling uncertainty. Results are from a replay of completed records.

Commit: {commit}    Run date: {today}
"""

team_msg = f"""DS1 UPDATE
Data: {len(df):,} CASH_OUT/TRANSFER rows, {int(df['isFraud'].sum()):,} fraud ({df['isFraud'].mean()*100:.4f}%).
Fraud by type:
{type_lines}
Split by time: train/validation/test as in README. Test scored once.
Threshold: {threshold:.5f} (target FPR {TARGET_FPR*100:.2f}%).
Test: recall {final['recall']}, precision {final['precision']}, FPR {final['FPR']}, F1 {final['F1']} (TP {final['TP']}, FP {final['FP']}, FN {final['FN']}, TN {final['TN']}).
Model input fields: type, amount, oldbalanceOrg, oldbalanceDest.
API derives: is_transfer = 1 if type == 'TRANSFER' else 0; amount_to_old_balance = amount / (oldbalanceOrg + 1.0); amount_exceeds_balance = 1 if amount > oldbalanceOrg else 0.
Feature order: see artifacts/model_meta.json. Hold = score >= threshold.
Files for you: artifacts/model.txt, model_meta.json, example_row.json, test_predictions.csv.
DECISION NEEDED: the abstract says 'ledger discrepancies', but the model cannot use that field. Reword, or say in the video that the demo replays completed records.
"""

with open(f'{ART}/README_model_section.md', 'w', encoding='utf-8') as f:
    f.write(readme)
with open(f'{ART}/team_message.txt', 'w', encoding='utf-8') as f:
    f.write(team_msg)

print(team_msg)
print("Saved README section and team message in:", ART)

DS1 UPDATE
Data: 2,770,409 CASH_OUT/TRANSFER rows, 8,213 fraud (0.2965%).
Fraud by type:
  CASH_OUT: 4,116 fraud of 2,237,500 rows
  TRANSFER: 4,097 fraud of 532,909 rows
Split by time: train/validation/test as in README. Test scored once.
Threshold: 0.00446 (target FPR 0.50%).
Test: recall 1.0, precision 0.885, FPR 0.00433, F1 0.939 (TP 1654, FP 215, FN 0, TN 49417).
Model input fields: type, amount, oldbalanceOrg, oldbalanceDest.
API derives: is_transfer = 1 if type == 'TRANSFER' else 0; amount_to_old_balance = amount / (oldbalanceOrg + 1.0); amount_exceeds_balance = 1 if amount > oldbalanceOrg else 0.
Feature order: see artifacts/model_meta.json. Hold = score >= threshold.
Files for you: artifacts/model.txt, model_meta.json, example_row.json, test_predictions.csv.
DECISION NEEDED: the abstract says 'ledger discrepancies', but the model cannot use that field. Reword, or say in the video that the demo replays completed records.

Saved README section and team message in: artifacts


In [6]:
from sklearn.metrics import average_precision_score

print("CHECK A: is the fraud really only in CASH_OUT and TRANSFER? (whole file, before filtering)")
print(pd.read_csv(dataset_path, usecols=['type', 'isFraud']).groupby('type')['isFraud'].sum().to_string())

print("\nCHECK B: simple rule vs LightGBM on TEST (rule: amount equals the origin balance, i.e. full drain)")
def drain_rule(part):
    return ((part['oldbalanceOrg'] > 0) & ((part['amount'] - part['oldbalanceOrg']).abs() <= 1.0)).values
cmp = pd.DataFrame([
    {'method': 'rule: full drain', **metrics(y_test, drain_rule(test))},
    {'method': 'LightGBM at chosen threshold', **metrics(y_test, p_test >= threshold)},
])
print(cmp[['method', 'recall', 'precision', 'FPR', 'F1', 'TP', 'FP', 'FN']].to_string(index=False))

print("\nCHECK C: how coarse are the model scores, and does more boosting help? (VALIDATION data)")
print("Distinct score values from the chosen model:", len(np.unique(p_val)))
pos = int(train['isFraud'].sum()); neg = len(train) - pos
for n in [1, 10, 50, 150]:
    m = lgb.LGBMClassifier(objective='binary', n_estimators=n, learning_rate=0.05, num_leaves=31,
                           max_depth=6, scale_pos_weight=(neg / pos) * 0.1,
                           random_state=42, n_jobs=-1, verbose=-1)
    m.fit(train[HONEST_FEATURES], train['isFraud'])
    pv = m.predict_proba(val[HONEST_FEATURES])[:, 1]
    print(f"  {n:>3} rounds | validation average precision {average_precision_score(y_val, pv):.4f} | distinct scores {len(np.unique(pv))}")

CHECK A: is the fraud really only in CASH_OUT and TRANSFER? (whole file, before filtering)
type
CASH_IN        0
CASH_OUT    4116
DEBIT          0
PAYMENT        0
TRANSFER    4097

CHECK B: simple rule vs LightGBM on TEST (rule: amount equals the origin balance, i.e. full drain)
                      method  recall  precision     FPR     F1   TP  FP  FN
            rule: full drain  0.9649      1.000 0.00000 0.9822 1596   0  58
LightGBM at chosen threshold  1.0000      0.885 0.00433 0.9390 1654 215   0

CHECK C: how coarse are the model scores, and does more boosting help? (VALIDATION data)
Distinct score values from the chosen model: 16
    1 rounds | validation average precision 0.9834 | distinct scores 13
   10 rounds | validation average precision 0.7475 | distinct scores 329
   50 rounds | validation average precision 0.7075 | distinct scores 1811
  150 rounds | validation average precision 0.7646 | distinct scores 3774


In [9]:
print("CHECK D: what does each layer add on TEST? (rule = auto-hold tier, model-only = human review queue)")
rule_t = drain_rule(test)
model_t = (p_test >= threshold)
yt = y_test.astype(bool)

def tier_line(name, mask):
    n = int(mask.sum()); tp = int((mask & yt).sum())
    return {'tier': name, 'holds': n, 'true_fraud': tp, 'honest_held': n - tp,
            'precision': round(tp / n, 4) if n else 0.0}

tiers = pd.DataFrame([
    tier_line('Tier 1: rule (auto-hold)', rule_t),
    tier_line('Tier 2: model-only (review queue)', model_t & ~rule_t),
    tier_line('Combined: rule OR model', rule_t | model_t),
])
print(tiers.to_string(index=False))

missed = test[yt & ~rule_t]
print("\nFraud the rule misses:", len(missed))
print(missed['type'].value_counts().to_string())
print("  origin balance is 0         :", int((missed['oldbalanceOrg'] == 0).sum()))
print("  amount below origin balance :", int((missed['amount'] < missed['oldbalanceOrg']).sum()))
print("  amount above origin balance :", int((missed['amount'] > missed['oldbalanceOrg']).sum()))
print(missed[['amount', 'oldbalanceOrg', 'oldbalanceDest']].describe().round(2).to_string())
tiers.to_csv(f'{ART}/rule_vs_model_tiers.csv', index=False)

CHECK D: what does each layer add on TEST? (rule = auto-hold tier, model-only = human review queue)
                             tier  holds  true_fraud  honest_held  precision
         Tier 1: rule (auto-hold)   1596        1596            0     1.0000
Tier 2: model-only (review queue)    273          58          215     0.2125
          Combined: rule OR model   1869        1654          215     0.8850

Fraud the rule misses: 58
type
TRANSFER    50
CASH_OUT     8
  origin balance is 0         : 8
  amount below origin balance : 50
  amount above origin balance : 0
           amount  oldbalanceOrg  oldbalanceDest
count        58.0           58.0           58.00
mean    8408884.0     19099170.0        24342.21
std     3633531.0     13774755.0       136811.47
min           0.0            0.0            0.00
25%    10000000.0     11885025.5            0.00
50%    10000000.0     15280485.5            0.00
75%    10000000.0     26905829.0            0.00
max    10000000.0     57316256.0   

In [10]:
rule_t = drain_rule(test)
model_t = (p_test >= threshold)
decision = np.where(rule_t, 'HOLD', np.where(model_t, 'REVIEW', 'RELEASE'))

out = test[['step', 'type', 'amount', 'oldbalanceOrg', 'oldbalanceDest']].copy()
out['y_true'] = y_test
out['score'] = p_test
out['rule_fired'] = rule_t.astype(int)
out['decision'] = decision
out.to_csv(f'{ART}/test_predictions.csv', index=False)

with open(f'{ART}/model_meta.json') as f:
    meta = json.load(f)
meta['rule'] = 'fire if oldbalanceOrg > 0 and abs(amount - oldbalanceOrg) <= 1.0'
meta['decision_logic'] = 'rule fires -> HOLD; else score >= threshold -> REVIEW; else RELEASE'
meta['tier_results_test'] = json.loads(tiers.to_json(orient='records'))
with open(f'{ART}/model_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

t1 = tiers.iloc[0]; t2 = tiers.iloc[1]; tc = tiers.iloc[2]
section = f"""
**Two-tier decision (test set):** Tier 1 rule (hold when the amount equals the origin balance): {int(t1['holds']):,} holds, {int(t1['true_fraud']):,} fraud, {int(t1['honest_held']):,} honest held. Tier 2 (model score at the chosen threshold, rule did not fire): {int(t2['holds']):,} flagged for human review, {int(t2['true_fraud']):,} fraud, {int(t2['honest_held']):,} honest (precision {t2['precision']}). Combined: {int(tc['true_fraud']):,} of {int(y_test.sum()):,} fraud found, {int(tc['honest_held']):,} honest held. The rule's perfect precision reflects how PaySim generates fraud and should not be expected on real data. Counts are at the test set's fraud rate of {y_test.mean()*100:.2f}%.
"""
with open(f'{ART}/README_model_section.md', 'a', encoding='utf-8') as f:
    f.write(section)

print(pd.crosstab(out['decision'], out['y_true'], rownames=['decision'], colnames=['actual fraud']))
print("Updated: test_predictions.csv, model_meta.json, README_model_section.md")

actual fraud      0     1
decision                 
HOLD              0  1596
RELEASE       49417     0
REVIEW          215    58
Updated: test_predictions.csv, model_meta.json, README_model_section.md


In [24]:
print(os.path.abspath(ART))
print(os.listdir(ART))

c:\Users\DELL\Desktop\IBM\PROJECT TRUST SHIELD\artifacts
['example_row.json', 'honest_vs_leaky.csv', 'model.txt', 'model_meta.json', 'README_model_section.md', 'rule_vs_model_tiers.csv', 'team_message.txt', 'test_predictions.csv']
